In [ ]:
# =============================================
#                   IMPORT
# =============================================
from keras.optimizers                   import Adam
from keras.callbacks                    import EarlyStopping, ReduceLROnPlateau
from preProcessingData                  import PreProcessData
from model.systemModel1                  import SystemModel
from model.systemModel1                  import SubclassModelCheckpoint

from tqdm                               import tqdm
from datetime                           import datetime
from prettytable                        import PrettyTable
import plotly.graph_objects             as go
import tensorflow                       as tf
import numpy                            as np 
import pandas                           as pd
import matplotlib.pyplot                as plt
import os, warnings
import random
import json


# ====================================
# Set debug level and 
# ------------------------------------
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"  
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'  # or any {'0', '1', '2'}
warnings.filterwarnings('ignore') 

np.random.seed(1)
random.seed(1)
tf.random.set_seed(1)

# ====================================
# Config
# ------------------------------------
with open('utility/modelConfig.json', 'r') as configFile:
    config = json.load(configFile)
myConfig   = config["system"]

In [ ]:
preData                         = PreProcessData(config = myConfig)                    # Preprocessing data
cr1, cr2, cv2                   = preData.loadH5File(name = "criticalAllData", keys = ["cr1", "cr2", "cv2"])                           # Load the data from critical file

withIDCr1, withIDCr2, withIDCv2, IdSys = preData.attachIDPerEvent(df = [cr1,cr2,cv2])

# Add the time to the data
# -----------------------------
# Normalize
normalizedCr1           = preData.normalizeBuilinFunction(withIDCr1, headline = preData.crusher1Value)  
normalizedCr2           = preData.normalizeBuilinFunction(withIDCr2, headline = preData.crusher2Value)  
normalizedCv2           = preData.normalizeBuilinFunction(withIDCv2, headline = preData.conveyorValue)  

# -----------------------------
# Add the Cycle and RUL to 
# each component
withTimeDataCr1         = preData.addCycleAndRul(normalizedCr1)                        # Add the time to the data
withTimeDataCr2         = preData.addCycleAndRul(normalizedCr2)                        # Add the time to the data
withTimeDataCv2         = preData.addCycleAndRul(normalizedCv2)                        # Add the time to the data


# -----------------------------
# Extract before split
dataCr1                 = withTimeDataCr1[preData.crusher1Value]
dataCr2                 = withTimeDataCr2[preData.crusher2Value]
dataCv2                 = withTimeDataCv2[preData.conveyorValue]


In [ ]:
model       = SystemModel(myConfig) 
model.build(input_shape=[(None, myConfig["seqLength"], myConfig["inCr1"]), (None, myConfig["seqLength"], myConfig["inCr2"]), (None, myConfig["seqLength"], myConfig["inCv2"])])  # Batch size is None, feature size is 32()

# model.build(input_shape=[(None, myConfig["seqLength"], myConfig["inCr1"]), (None, myConfig["seqLength"], myConfig["inCr2"]), (None, myConfig["seqLength"], myConfig["inCv2"])])  # Batch size is None, feature size is 32()
model.summary()
model.load_weights(model_dir = "checkpoint/bestModel")


# (522340, 60, 32) (522340, 1) (522340, 60, 30) (522340, 1) (522340, 60, 2) (522340, 1) (522340, 1)

In [ ]:
results            = {}
results["cr1"]     = []
results["cr2"]     = []
results["cv2"]     = []
results["sys"]     = []
results["rul"]     = []

probsAll        = []
binaryMaskAll   = []
trajectID = IdSys.astype(int)
for idx in range(trajectID.max()):
    print(idx)
    count = (trajectID == (idx + 1)).sum() 
    if count < preData.config['seqLength']:
        continue
    seqDataCr1,seqDataCr2,seqDataCv2         = preData.getTensorData(dataList   =    [dataCr1, dataCr2, dataCv2], 
                                                                    trajectId   = trajectID,
                                                                    seqLength  = preData.config['seqLength'], 
                                                                    overlap    = preData.config['seqLength']-1,
                                                                    getAtIdx   = idx + 1,
                                                                    typeInput  = "listOfPandas")
    
    zCr1, zCr2,  zCv2,                              \
    binaryMaskCr1, binaryMaskCr2, binaryMaskCv2,    \
    probsMaskCr1, probsMaskCr2, probsMaskCv2,       \
    hiSystem, predictedRul                          = model.getHIandRUL([seqDataCr1,seqDataCr2,seqDataCv2])
    results["cr1"].append(zCr1)
    results["cr2"].append(zCr2)
    results["cv2"].append(zCv2)
    results["sys"].append(hiSystem)
    results["rul"].append(predictedRul)

### MASK

##### Crusher1

In [ ]:
ssTable                 = PrettyTable()
ssTable.field_names     = ["No","sensorName","probability", "binary"]
print(f"The number of selected sensor {sum(binaryMaskCr1)}")
for idx,idxSS in enumerate(preData.crusher1Value):
    ssTable.add_row([idx,idxSS,probsMaskCr1[idx].numpy(),binaryMaskCr1[idx].numpy()])

print(ssTable)


##### Crusher 2

In [ ]:
ssTable                 = PrettyTable()
ssTable.field_names     = ["No","sensorName","probability", "binary"]
print(f"The number of selected sensor {sum(binaryMaskCr2)}")
for idx,idxSS in enumerate(preData.crusher2Value):
    ssTable.add_row([idx,idxSS,probsMaskCr2[idx].numpy(),binaryMaskCr2[idx].numpy()])

print(ssTable)

##### ConveyorBelt 2

In [ ]:
ssTable                 = PrettyTable()
ssTable.field_names     = ["No","sensorName","probability", "binary"]
print(f"The number of selected sensor {sum(binaryMaskCv2)}")
for idx,idxSS in enumerate(preData.conveyorValue):
    ssTable.add_row([idx,idxSS,probsMaskCv2[idx].numpy(),binaryMaskCv2[idx].numpy()])

print(ssTable)

### Health indicators

##### Crusher 1

In [ ]:
fig = {}
for idxResults in results.keys():
    fig[idxResults]     = go.Figure()
    current_time        = 0

    for idx, signal in enumerate(results[idxResults]):
        signal          = tf.squeeze(signal).numpy()
        time_vector     = np.arange(current_time, current_time + len(signal))
        current_time    += len(signal)
        
        # Add each signal as a separate trace
        fig[idxResults].add_trace(go.Scatter(
                                                x   = time_vector,
                                                y   = signal,
                                                mode= 'lines',
                                                name= f'Signal {idx + 1}'  # Label each signal for legend toggling
        ))
    # Customize layout for readability
    fig[idxResults].update_layout(
                                    title       = idxResults,
                                    xaxis_title = "Time (arbitrary units)",
                                    yaxis_title = "Signal Value",
                                    template    = "plotly_white",
                                    showlegend  = True  # Show legend to allow toggling each signal
    )
    # Display the plot
    fig[idxResults].show()


##### Crusher 2

In [ ]:
fig1             = go.Figure()
current_time    = 0

for idx, signal in enumerate(results["cr2"]):
    signal          = tf.squeeze(signal).numpy()
    time_vector     = np.arange(current_time, current_time + len(signal))
    current_time    += len(signal)
    
    # Add each signal as a separate trace
    fig1.add_trace(go.Scatter(
        x=time_vector,
        y=signal,
        mode='lines',
        name=f'Signal {idx + 1}'  # Label each signal for legend toggling
    ))

# Customize layout for readability
fig1.update_layout(
    title="Continuous Time Series Plot with Toggle",
    xaxis_title="Time (arbitrary units)",
    yaxis_title="Signal Value",
    template="plotly_white",
    showlegend=True  # Show legend to allow toggling each signal
)

# Display the plot
fig1.show()
